# RentFleet — ANPR marocain v2 : smoke détecteur + OCR

Ce notebook teste sur GPU le détecteur privé v1.2 déjà sélectionné, puis lit le matricule avec le modèle officiel `arabic_PP-OCRv5_mobile_rec`. PyTorch et PaddleOCR utilisent deux environnements Python et deux processus isolés sur la même T4. Il applique plusieurs marges, un contraste conservateur et une rectification géométrique non générative.

Le résultat est **consultatif** : aucune plaque n'est enregistrée dans le SaaS, aucun véhicule n'est créé ou modifié et une lecture ambiguë est rejetée.


## Contrat de cette exécution

- échantillon technique déterministe issu d'une source de développement déjà consommée ;
- aucun accès au futur test indépendant ;
- aucune affirmation de précision à partir d'un smoke non annoté ;
- checkpoint et sélection vérifiés par SHA-256 dans le Drive privé ;
- prédictions et matricules conservés uniquement dans le Drive privé ;
- les variantes d'une même photo ne comptent jamais comme plusieurs vues indépendantes.


In [ ]:
#@title 1. Monter Drive et vérifier le GPU Torch
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path, PurePosixPath
import hashlib, json, os, random, shutil, subprocess, sys, zipfile
import torch

assert torch.cuda.is_available(), 'Sélectionnez Exécution > Modifier le type d’exécution > GPU.'
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
})


In [ ]:
#@title 2. Récupérer la branche scientifique GitHub
REPOSITORY = 'https://github.com/getibplay-cmyk/pfe.git'
GIT_REF = 'science/moroccan-anpr-v2'
REPO_DIR = Path('/content/pfe')

if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--filter=blob:none', '--branch', GIT_REF,
        '--single-branch', REPOSITORY, str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GIT_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--force', GIT_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{GIT_REF}'], check=True)
GIT_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
print({'git_ref': GIT_REF, 'git_sha': GIT_SHA})


In [ ]:
#@title 3. Installer PaddlePaddle GPU et PaddleOCR dans un venv isolé
cuda_version = str(torch.version.cuda or '')
if cuda_version.startswith('11.'):
    paddle_index = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
elif cuda_version.startswith('12.'):
    paddle_index = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
else:
    raise RuntimeError(f'Génération CUDA Colab non prise en charge: {cuda_version!r}')

OCR_VENV = Path('/content/venvs/rentfleet-paddleocr-v2')
OCR_PYTHON = OCR_VENV / 'bin/python'
OCR_VENV.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
if not OCR_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'venv', str(OCR_VENV)], check=True)
subprocess.run([
    str(OCR_PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
    'paddlepaddle-gpu==3.3.0', '--index-url', paddle_index,
], check=True)
subprocess.run([
    str(OCR_PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
    '--requirement',
    str(REPO_DIR / 'scripts/intelligence/requirements-vehicle-plate-colab.txt'),
], check=True)
subprocess.run([str(OCR_PYTHON), '-m', 'pip', 'check'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

os.environ['PADDLE_PDX_MODEL_SOURCE'] = 'BOS'
verification_code = '''
import json, paddle, paddleocr
paddle.utils.run_check()
assert paddle.is_compiled_with_cuda()
assert paddle.device.cuda.device_count() >= 1
print(json.dumps({
    'paddle': paddle.__version__,
    'paddleocr': paddleocr.__version__,
    'paddle_gpu_count': paddle.device.cuda.device_count(),
}))
'''
subprocess.run([str(OCR_PYTHON), '-c', verification_code], check=True, env=os.environ.copy())
print({'wheel_index': paddle_index, 'ocr_python': str(OCR_PYTHON), 'isolated': True})


In [ ]:
#@title 4. Chemins privés et paramètres du smoke
DRIVE_ROOT = Path('/content/drive/MyDrive/RentFleet_PFE/S7_vehicle_vision_assistant')
PRIVATE_CHECKPOINT = (
    DRIVE_ROOT / 'modeles_prives/S7_06_detection_multidomaine_v1.2.0/'
    'S7_06_FASTER_RCNN_RESNET50_FPN_V2_MULTIDOMAIN_v1.2.0.pt'
)
PRIVATE_SELECTION = (
    DRIVE_ROOT / 'metriques_figures/S7_06_detection_multidomaine_v1.2.0/'
    'S7_06_MULTIDOMAIN_MODEL_SELECTION_v1.2.0.json'
)
PRIMARY_ARCHIVE = (
    DRIVE_ROOT / 'donnees_brutes_privees/S7_06_detection_plaques_marocaines/'
    'S7_06_KAGGLE_MOROCCAN_PLATES_CC0_v2.zip'
)
SOURCE_AUDIT = (
    DRIVE_ROOT / 'metriques_figures/S7_06_detection_multidomaine_v1.2.0/'
    'S7_06_MULTIDOMAIN_SOURCE_AUDIT_v1.2.0.json'
)
SERIES_MAPPING = (
    DRIVE_ROOT / 'gouvernance_et_preinscription/'
    'S7_ANPR_SERIES_MAPPING_VERIFIED_v2.0.0.json'
)
LOCAL_CHECKPOINT = Path('/content/anpr_detector_v1.2.0.pt')
LOCAL_ARCHIVE = Path('/content/moroccan_plates_cc0_v2.zip')
LOCAL_INPUT = Path('/content/anpr_v2_smoke_input')
MAX_IMAGES = 12 #@param {type:'integer'}
RUN_ID = 'anpr_v2_smoke_ppocrv5_seed20260825_run01' #@param {type:'string'}
OUTPUT = DRIVE_ROOT / 'modeles' / RUN_ID

for required in (PRIVATE_CHECKPOINT, PRIVATE_SELECTION, PRIMARY_ARCHIVE, SOURCE_AUDIT):
    assert required.is_file(), f'Artefact privé absent: {required}'
assert 1 <= MAX_IMAGES <= 24, 'Le smoke est limité à 24 images.'
print({
    'max_images': MAX_IMAGES,
    'run_id': RUN_ID,
    'official_bilingual_mapping_available': SERIES_MAPPING.is_file(),
    'output': str(OUTPUT),
})


In [ ]:
#@title 5. Vérifier et copier les artefacts, puis préparer l'échantillon
sys.path.insert(0, str(REPO_DIR))
from scripts.intelligence.vehicle_plate.protocol import file_sha256

selection = json.loads(PRIVATE_SELECTION.read_text(encoding='utf-8'))
source_audit = json.loads(SOURCE_AUDIT.read_text(encoding='utf-8'))
expected_model_sha = selection['selected_model_sha256']
expected_archive_sha = source_audit['sources']['primary_archive_sha256']

def copy_verified(source, target, expected_sha):
    if target.is_file() and file_sha256(target) == expected_sha:
        return 'reused'
    shutil.copy2(source, target)
    actual = file_sha256(target)
    assert actual == expected_sha, f'Empreinte différente après copie: {source.name}'
    return 'copied'

model_copy = copy_verified(PRIVATE_CHECKPOINT, LOCAL_CHECKPOINT, expected_model_sha)
archive_copy = copy_verified(PRIMARY_ARCHIVE, LOCAL_ARCHIVE, expected_archive_sha)

if LOCAL_INPUT.exists():
    shutil.rmtree(LOCAL_INPUT)
LOCAL_INPUT.mkdir(parents=True)
with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f'Archive corrompue: {bad_member}'
    image_members = sorted(
        name for name in archive.namelist()
        if not name.endswith('/')
        and PurePosixPath(name).suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
        and '__MACOSX' not in PurePosixPath(name).parts
    )
    training_members = [
        name for name in image_members
        if any(part.lower() in {'train', 'training'} for part in PurePosixPath(name).parts)
    ]
    population = training_members or image_members
    assert len(population) >= MAX_IMAGES
    selected = sorted(random.Random(20260825).sample(population, MAX_IMAGES))
    for name in selected:
        suffix = PurePosixPath(name).suffix.lower()
        safe_name = hashlib.sha256(name.encode('utf-8')).hexdigest()[:16] + suffix
        (LOCAL_INPUT / safe_name).write_bytes(archive.read(name))

print({
    'checkpoint': model_copy,
    'archive': archive_copy,
    'source_role': 'development_smoke_only',
    'prepared_images': len(list(LOCAL_INPUT.iterdir())),
    'labels_opened': False,
})


In [ ]:
#@title 6. Exécuter le smoke détecteur + OCR
command = [
    sys.executable,
    str(REPO_DIR / 'scripts/intelligence/vehicle_plate/colab_smoke.py'),
    '--input-dir', str(LOCAL_INPUT),
    '--checkpoint', str(LOCAL_CHECKPOINT),
    '--selection', str(PRIVATE_SELECTION),
    '--ocr-python', str(OCR_PYTHON),
    '--output-dir', str(OUTPUT),
    '--max-images', str(MAX_IMAGES),
]
if SERIES_MAPPING.is_file():
    command.extend(['--series-mapping', str(SERIES_MAPPING)])
subprocess.run(command, cwd=REPO_DIR, check=True, env=os.environ.copy())

summary = json.loads((OUTPUT / 'SMOKE_COMPLETE.json').read_text(encoding='utf-8'))
assert summary['status'] == 'smoke_complete_not_qualified'
assert summary['qualification_claim'] is False
assert summary['final_test_opened'] is False
print(json.dumps({
    'status': summary['status'],
    'counts': summary['counts'],
    'metrics': summary['metrics'],
    'timings_seconds': summary['timings_seconds'],
    'bilingual_mapping_verified': summary['ocr']['bilingual_mapping_verified'],
}, ensure_ascii=False, indent=2))


## Lecture du résultat

Un `SMOKE_COMPLETE.json` valide prouve seulement que le checkpoint, le GPU, la détection, les crops et l'OCR fonctionnent ensemble. L'échantillon non annoté ne permet pas de calculer la précision.

La prochaine étape est un lot RentFleet consenti, annoté et séparé en développement/calibration, puis le fine-tuning du recognizer. Le futur test indépendant reste fermé jusqu'au gel complet du modèle, du seuil et du décodeur.
